In [1]:
import polars as pl 
import numpy as np 
import cobra as cb
from pathlib import Path
from cobra.io.mat import load_matlab_model
from cobra.util import constraint_matrices, nullspace

In [3]:
moodel_path = Path("../../outputs/clus_vs_disp_redo/Disp.mat")

moodel = load_matlab_model(moodel_path)

No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, g, i, l, m, n, r, x


In [24]:
moodel_path.stem

'Disp'

In [4]:
moodel

Name,model
Memory address,1d0e641e8d0
Number of metabolites,5160
Number of reactions,8544
Number of genes,2248
Number of groups,99
Objective expression,1.0*biomass_maintenance - 1.0*biomass_maintenance_reverse_95d2f
Compartments,"c, l, m, r, e, x, n, g, i"


In [16]:
moodel.reactions.get_by_id('biomass_maintenance').lower_bound = 25


In [17]:
moodel.reactions.get_by_id('biomass_maintenance')

Reaction identifier,biomass_maintenance
Name,Biomass maintenance reaction without replication precursors
Memory address,0x1d0ee47e600
Stoichiometry,0.50563 ala_L[c] + 0.35926 arg_L[c] + 0.27942 asn_L[c] + 0.35261 asp_L[c] + 20.7045 atp[c] + 0.020401 chsterol[c] + 0.011658 clpn_hs[c] + 0.039036 ctp[c] + 0.046571 cys_L[c] + 0.27519 g6p[c] +... 0.50563 L-Alanine + 0.35926 L-Arginine + 0.27942 L-Asparagine + 0.35261 L-Aspartate + 20.7045 Adenosine Triphosphate + 0.020401 Cholesterol + 0.011658 Cardiolipin + 0.039036...
GPR,
Lower bound,25
Upper bound,1000.0


In [23]:
growths = pl.read_csv("../../outputs/clus_vs_disp_redo/characterization/growths.csv")
{condition.name:condition[0] for condition in growths.iter_columns()}

{'Clus': 447.112640821751, 'Disp': 435.440291822407}

In [4]:
import sammi

In [25]:
subsystem = pl.read_csv("../../data/recon_reaction_names.csv")
subsystem

Reaction,subSystem
str,str
"""10FTHF5GLUtl""","""Transport, lysosomal"""
"""10FTHF5GLUtm""","""Transport, mitochondrial"""
"""10FTHF6GLUtl""","""Transport, lysosomal"""
"""10FTHF6GLUtm""","""Transport, mitochondrial"""
"""10FTHF7GLUtl""","""Transport, lysosomal"""
…,…
"""CYOR_u10mi""","""Oxidative phosphorylation"""
"""Htmi""","""Transport, mitochondrial"""
"""NADH2_u10mi""","""Oxidative phosphorylation"""


In [26]:
reactions = pl.read_csv("../../results/sampling_results/clus_vs_disp/differential reactions.csv")
reactions = reactions.join(subsystem, 'Reaction')
reactions.sort('fc', 'subSystem')

Reaction,fc,pval,subSystem
str,f64,i64,str
"""EX_proglnpro[e]""",-469.180042,0,"""Exchange/demand reaction"""
"""TRPSERTYRr""",-122.277109,0,"""Peptide metabolism"""
"""TRPSERTYRt""",-122.277109,0,"""Transport, extracellular"""
"""INSTt4""",-98.501769,0,"""Transport, extracellular"""
"""EX_valtrpval[e]""",-79.181871,0,"""Exchange/demand reaction"""
…,…,…,…
"""VALTRPVALt""",79.181871,0,"""Transport, extracellular"""
"""EX_trpsertyr[e]""",122.277109,0,"""Exchange/demand reaction"""
"""EX_amp[e]""",178.972347,0,"""Exchange/demand reaction"""


In [27]:
reactions.cast(pl.Utf8)

Reaction,fc,pval,subSystem
str,str,str,str
"""10FTHFtm""","""0.8945530180773418""","""0""","""Transport, mitochondrial"""
"""34DHOXPEGOX""","""0.8248386406955573""","""0""","""Tyrosine metabolism"""
"""34DHOXPEGt""","""0.8248386406955573""","""0""","""Transport, extracellular"""
"""34DHPHAMT""","""0.9686315273615296""","""0""","""Tyrosine metabolism"""
"""AATAi""","""-2.9422058280226975""","""0""","""Lysine metabolism"""
…,…,…,…
"""4HATVACIDthc""","""-9.393254268532809""","""0""","""Drug metabolism"""
"""ALLOP1tu""","""-0.9772530054436985""","""0""","""Drug metabolism"""
"""CRVSM24tev""","""-5.2708440178118545""","""0""","""Drug metabolism"""


In [7]:
reactions.filter(pl.col("fc")>0).sort('fc',)

Reaction,fc,pval,subSystem
str,f64,i64,str
"""NDPK3n""",0.820024,0,"""Nucleotide interconversion"""
"""r2164""",0.822304,0,"""Transport, extracellular"""
"""r1942""",0.822876,0,"""Transport, extracellular"""
"""IDPtn""",0.824335,0,"""Transport, nuclear"""
"""NDPK9n""",0.824335,0,"""Nucleotide interconversion"""
…,…,…,…
"""VALTRPVALr""",79.181871,0,"""Peptide metabolism"""
"""EX_trpsertyr[e]""",122.277109,0,"""Exchange/demand reaction"""
"""EX_amp[e]""",178.972347,0,"""Exchange/demand reaction"""


In [8]:
disp_rxn = reactions.filter(pl.col("fc")>0).sort('fc',descending=True)['Reaction'].to_list()

In [9]:
plot_data = (
    reactions
    .filter(pl.col('fc')>0) # Filter for condition 2
    .group_by('subSystem')
    .agg(pl.col('Reaction'), pl.col('fc'))
    .with_columns(subsystem_size = pl.col('Reaction').list.len())
    .sort('subsystem_size',descending=True)
)
plot_data[:10]

subSystem,Reaction,fc,subsystem_size
str,list[str],list[f64],u32
"""Transport, extracellular""","[""34DHOXPEGt"", ""ALAGLNexR"", … ""M02155tr""]","[0.824839, 19.667735, … 0.913281]",209
"""Exchange/demand reaction""","[""EX_34dhoxpeg[e]"", ""EX_4nph[e]"", … ""DM_hdca24g[c]""]","[0.824839, 2.318831, … 0.921107]",94
"""Peptide metabolism""","[""ALALYSTHRr"", ""ARGALATHRr"", … ""VALVALr""]","[1.345548, 1.015598, … 0.973971]",70
"""Transport, mitochondrial""","[""10FTHFtm"", ""CITRtm"", … ""HMR_2764""]","[0.894553, 5.015803, … 20.474242]",22
"""Nucleotide interconversion""","[""CYTK10n"", ""CYTK11"", … ""NDPK3""]","[2.040652, 2.117552, … 2.11114]",17
"""Fatty acid oxidation""","[""C161CRN2t"", ""C204CRNt"", … ""HMR_2660""]","[9.208795, 2.198795, … 0.964764]",11
"""Transport, endoplasmic reticul…","[""AHCYStr"", ""BILGLCURtr"", … ""HMR_7949""]","[0.825966, 0.993357, … 1.625553]",8
"""Miscellaneous""","[""r0409"", ""r0587"", … ""EX_gm1_hs[e]""]","[3.363584, 2.391333, … 3.057955]",4
"""Cholesterol metabolism""","[""LCAT25e"", ""LCAT43e"", … ""LCAT6e""]","[0.944243, 0.832966, … 0.825992]",4


In [10]:
plot_list = []
for  row in plot_data.iter_rows():
    plot_list.append(
        sammi.parser(row[0],row[1], row[2])
    )

In [11]:
sammi.plot(moodel, plot_list,opts=sammi.options("Disp.html"))

c:\Users\loksh\OneDrive - smail.iitm.ac.in\Projects\NUS collaboration\conda_env\Lib\site-packages\cobra\core\group.py:147: UserWarning: need to pass in a list
  warn("need to pass in a list")
c:\Users\loksh\OneDrive - smail.iitm.ac.in\Projects\NUS collaboration\conda_env\Lib\site-packages\cobra\core\metabolite.py:191: UserWarning: The element 'L' does not appear in the periodic table
  warn(f"The element {e} does not appear in the periodic table")
c:\Users\loksh\OneDrive - smail.iitm.ac.in\Projects\NUS collaboration\conda_env\Lib\site-packages\cobra\core\metabolite.py:191: UserWarning: The element 'X' does not appear in the periodic table
  warn(f"The element {e} does not appear in the periodic table")


In [12]:
len(disp_rxn)

478

In [13]:
model2 = cb.io.load_matlab_model("../../outputs/clus_vs_disp/Clus.mat")

No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, g, i, l, m, n, r, x


In [14]:
model2

Name,model
Memory address,2214ec13f50
Number of metabolites,5162
Number of reactions,8545
Number of genes,2248
Number of groups,99
Objective expression,0
Compartments,"c, l, m, r, e, x, n, g, i"


In [15]:
plot_data_clus = (
    reactions
    .filter(pl.col('fc')<0) # Filter for condition 2
    .group_by('subSystem')
    .agg(pl.col('Reaction'), pl.col('fc'))
    .with_columns(subsystem_size = pl.col('Reaction').list.len())
    .sort('subsystem_size',descending=True)
)
plot_data_clus[:10]

subSystem,Reaction,fc,subsystem_size
str,list[str],list[f64],u32
"""Transport, extracellular""","[""ALADGLNexR"", ""BILDGLCURt"", … ""UDCHOLt2""]","[-2.15898, -1.770073, … -0.882105]",254
"""Exchange/demand reaction""","[""EX_ach[e]"", ""EX_gd1c_hs[e]"", … ""EX_M00260[e]""]","[-6.637655, -3.485561, … -3.21637]",99
"""Peptide metabolism""","[""ARGALAPHEr"", ""ARGCYSSERr"", … ""VALTRPPHEr""]","[-3.446415, -6.209728, … -0.924269]",65
"""Transport, mitochondrial""","[""ADNtm"", ""FUMtm"", … ""HMR_2748""]","[-0.837546, -1.618242, … -1.089104]",20
"""Nucleotide interconversion""","[""ADK3m"", ""CYTK12"", … ""NTD6""]","[-2.680437, -1.071177, … -0.856806]",18
"""Fatty acid oxidation""","[""C161CPT2"", ""C180CRNt"", … ""HMR_2608""]","[-0.966403, -0.89699, … -5.338047]",14
"""Transport, endoplasmic reticul…","[""AMETr"", ""BILDGLCURtr"", … ""HMR_9634""]","[-0.825966, -1.734846, … -1.286181]",9
"""Transport, peroxisomal""","[""ARACHDCOAtx"", ""TMNDNCCOAtx"", … ""HMR_9680""]","[-0.889019, -2.754146, … -1.092843]",5
"""Miscellaneous""","[""r0242"", ""RE0936E"", … ""EX_5cysgly34dhphe[e]""]","[-3.403507, -4.716253, … -1.538614]",5


In [16]:
plot_list_clus = []
for  row in plot_data_clus.iter_rows():
    plot_list_clus.append(
        sammi.parser(row[0],row[1], row[2])
    )

In [18]:
sammi.plot(model2, plot_list_clus, opts=sammi.options("Clus.html"))

In [19]:
plot_data

subSystem,Reaction,fc,subsystem_size
str,list[str],list[f64],u32
"""Transport, extracellular""","[""34DHOXPEGt"", ""ALAGLNexR"", … ""M02155tr""]","[0.824839, 19.667735, … 0.913281]",209
"""Exchange/demand reaction""","[""EX_34dhoxpeg[e]"", ""EX_4nph[e]"", … ""DM_hdca24g[c]""]","[0.824839, 2.318831, … 0.921107]",94
"""Peptide metabolism""","[""ALALYSTHRr"", ""ARGALATHRr"", … ""VALVALr""]","[1.345548, 1.015598, … 0.973971]",70
"""Transport, mitochondrial""","[""10FTHFtm"", ""CITRtm"", … ""HMR_2764""]","[0.894553, 5.015803, … 20.474242]",22
"""Nucleotide interconversion""","[""CYTK10n"", ""CYTK11"", … ""NDPK3""]","[2.040652, 2.117552, … 2.11114]",17
…,…,…,…
"""Pyrimidine catabolism""","[""D3AIBTm""]",[0.855777],1
"""Tryptophan metabolism""","[""RE2349C""]",[1.813332],1
"""C5-branched dibasic acid metab…","[""MECOAS1m""]",[0.859373],1


In [20]:
plot_data_clus

subSystem,Reaction,fc,subsystem_size
str,list[str],list[f64],u32
"""Transport, extracellular""","[""ALADGLNexR"", ""BILDGLCURt"", … ""UDCHOLt2""]","[-2.15898, -1.770073, … -0.882105]",254
"""Exchange/demand reaction""","[""EX_ach[e]"", ""EX_gd1c_hs[e]"", … ""EX_M00260[e]""]","[-6.637655, -3.485561, … -3.21637]",99
"""Peptide metabolism""","[""ARGALAPHEr"", ""ARGCYSSERr"", … ""VALTRPPHEr""]","[-3.446415, -6.209728, … -0.924269]",65
"""Transport, mitochondrial""","[""ADNtm"", ""FUMtm"", … ""HMR_2748""]","[-0.837546, -1.618242, … -1.089104]",20
"""Nucleotide interconversion""","[""ADK3m"", ""CYTK12"", … ""NTD6""]","[-2.680437, -1.071177, … -0.856806]",18
…,…,…,…
"""Pentose phosphate pathway""","[""HMR_4592""]",[-1.820602],1
"""Glycerophospholipid metabolism""","[""r0480""]",[-0.837009],1
"""NAD metabolism""","[""r0584""]",[-0.907573],1


In [23]:
flux = pl.read_parquet(
    "../../outputs/sampling/clus_vs_disp/common.parquet"
)
flux.sample(10)

10FTHF7GLUtl,10FTHF7GLUtm,10FTHFtl,10FTHFtm,11DOCRTSLtm,11DOCRTSLtr,11DOCRTSTRNtm,11DOCRTSTRNtr,13DAMPPOX,24_25DHVITD2t,24_25DHVITD2tm,24_25DHVITD3t,24_25DHVITD3tm,24_25VITD2Hm,24_25VITD3Hm,24NPHte,25HVITD2tin_m,25HVITD3tin_m,25VITD2Hm,25VITD3Hm,2AMACHYD,2AMACSULT,2AMADPTm,2DR1PP,2HBO,2HBt2,2HCO3_NAt,2OXOADOXm,2OXOADPTm,34DHOXPEGOX,34DHOXPEGt,34DHPHAMT,34DHPHEt,34DHPLACOX,34DHPLACOX_NADP_,34DHXMANDACOX,34DHXMANDACOX_NADP_,…,EX_M00008[e],EX_M00117[e],EX_M00260[e],EX_M00315[e],EX_M01235[e],EX_M02053[e],EX_M02613[e],EX_M02745[e],EX_C01601[e],EX_M02108[e],EX_M03117[e],EX_M03134[e],EX_M01111[e],EX_M01872[e],EX_M01870[e],EX_ditp[e],EX_hnifedipine[e],EX_adpman[e],EX_rbl_D[e],EX_M01966[e],EX_M02155[e],EX_M02837[e],EX_M01881[e],EX_M03131[e],EX_n5m2masn[e],sink_4abut[l],DM_4glu56dihdind[c],DM_ind56qn[c],DM_cbl2[m],DM_btn[m],DCMPtm,ATPS4mi,CYOR_u10mi,Htmi,NADH2_u10mi,CYOOm3i,experiment
f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,str
4.159247,2.592376,-4.159247,152.69899,3.284055,-8.77896,1.660412,-2.689472,0.987604,6.889639,6.889639,6.580947,6.580947,6.889639,6.580947,-0.820043,17.32267,10.402963,10.433031,3.822016,32.429535,0.637448,52.18594,1.322812,-138.737625,-222.326996,-779.622559,12.315195,-616.144775,507.561462,507.561462,334.738129,-442.367584,228.419846,-725.931274,331.056549,-820.564453,…,-0.582337,7.462286,2.64189,3.451823,-2.474748,-5.51416,-9.61748,3.301331,-2.582828,-1.015545,-1.184744,-6.580539,377.246765,1.012743,1.486558,-2.652099,9.069139,-2.817703,-103.486931,759.7052,-105.422302,-217.813141,0.219082,2.736631,-1.486558,-205.675201,2.919042,68.698578,941.623596,189.684708,-689.918335,918.138,555.846619,547.894287,49.426796,449.838135,"""Disp"""
6.241165,1.91161,-6.241165,81.204636,4.161999,-1.016702,4.857914,-7.167997,5.638175,3.605225,3.605225,4.375394,4.375394,3.605225,4.375394,-1.587055,12.973705,7.581681,9.368481,3.206287,72.580673,1.042361,66.096626,0.901968,-97.750702,-308.46698,-691.656677,74.168129,-640.279724,473.65448,473.65448,196.587158,-387.918579,187.608841,-727.691833,347.715607,-801.15033,…,-1.666501,2.860154,-0.276959,7.844241,-0.727188,-2.477474,-8.341182,4.115492,1.835771,-2.498844,-1.490514,-0.352707,443.29361,1.079565,1.985034,-4.772085,14.149835,-12.674714,-54.896866,762.725708,-59.666294,-281.37207,0.370881,0.093047,-1.985034,-230.324326,1.111277,195.928787,909.395264,113.761391,-728.032715,910.863464,483.168518,564.982544,83.989731,484.950897,"""Disp"""
2.723069,0.060914,-2.723069,84.693726,3.970103,-3.764624,6.351401,-2.685149,12.943382,0.002511,0.002511,5.321191,5.321191,0.002511,5.321191,-2.744922,0.702638,5.50385,0.700127,0.182658,23.632605,1.879591,53.149319,3.400184,-68.717201,-259.60379,-759.359314,35.724312,-714.401733,577.297852,577.297852,272.17218,-426.071655,239.071884,-764.064087,261.891815,-816.887939,…,-5.719372,1.884688,-6.867784,15.700746,5.266262,3.885577,-13.364775,5.226439,0.716162,-0.719315,-0.521523,-1.968008,443.023987,0.211636,0.433661,-1.389375,25.860901,-10.401455,-63.403084,738.873596,-43.938061,-226.549561,0.109933,0.44742,-0.433661,-201.307938,0.35948,97.547371,936.539185,166.864471,-657.784729,932.76709,486.960846,626.084595,101.203079,501.124298,"""Disp"""
16.266958,5.384442,-16.266958,6.249233,5.119094,-2.816787,5.288664,-5.310087,2.989811,0.12655,0.12655,0.354425,0.354425,0.12655,0.354425,-30.913744,3.986014,3.972051,3.859463,3.617626,7.341931,2.579627,169.524887,7.041867,-236.596512,-164.34465,-372.983521,108.370964,-280.667511,49.020096,49.020096,13.458565,-354.432922,106.94413,-503.454712,167.429596,-198.269531,…,-2.911579,0.977021,0.477838,2.79827,0.069524,-1.773153,-4.633356,0.000832,11.016992,-0.627636,-0.253125,-0.994845,388.841003,1.6172,1.303927,-1.026538,8.112302,-11.371097,-20.762033,871.293457,-9.236247,-5

In [42]:
flux.pivot(
    on = 'experiment',
    index = pl.selectors.exclude('experiment'),
    values= pl.selectors.exclude('experiment'),
    aggregate_function='mean'
)

10FTHF7GLUtl,10FTHF7GLUtm,10FTHFtl,10FTHFtm,11DOCRTSLtm,11DOCRTSLtr,11DOCRTSTRNtm,11DOCRTSTRNtr,13DAMPPOX,24_25DHVITD2t,24_25DHVITD2tm,24_25DHVITD3t,24_25DHVITD3tm,24_25VITD2Hm,24_25VITD3Hm,24NPHte,25HVITD2tin_m,25HVITD3tin_m,25VITD2Hm,25VITD3Hm,2AMACHYD,2AMACSULT,2AMADPTm,2DR1PP,2HBO,2HBt2,2HCO3_NAt,2OXOADOXm,2OXOADPTm,34DHOXPEGOX,34DHOXPEGt,34DHPHAMT,34DHPHEt,34DHPLACOX,34DHPLACOX_NADP_,34DHXMANDACOX,34DHXMANDACOX_NADP_,…,EX_adpman[e]_Disp,EX_rbl_D[e]_Clus,EX_rbl_D[e]_Disp,EX_M01966[e]_Clus,EX_M01966[e]_Disp,EX_M02155[e]_Clus,EX_M02155[e]_Disp,EX_M02837[e]_Clus,EX_M02837[e]_Disp,EX_M01881[e]_Clus,EX_M01881[e]_Disp,EX_M03131[e]_Clus,EX_M03131[e]_Disp,EX_n5m2masn[e]_Clus,EX_n5m2masn[e]_Disp,sink_4abut[l]_Clus,sink_4abut[l]_Disp,DM_4glu56dihdind[c]_Clus,DM_4glu56dihdind[c]_Disp,DM_ind56qn[c]_Clus,DM_ind56qn[c]_Disp,DM_cbl2[m]_Clus,DM_cbl2[m]_Disp,DM_btn[m]_Clus,DM_btn[m]_Disp,DCMPtm_Clus,DCMPtm_Disp,ATPS4mi_Clus,ATPS4mi_Disp,CYOR_u10mi_Clus,CYOR_u10mi_Disp,Htmi_Clus,Htmi_Disp,NADH2_u10mi_Clus,NADH2_u10mi_Disp,CYOOm3i_Clus,CYOOm3i_Disp
f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
4.613117,0.346264,-4.613117,39.285633,0.035125,-0.000201,0.00141,-0.000578,0.400739,0.000211,0.000211,0.000287,0.000287,0.000211,0.000287,-0.002074,0.002911,0.001175,0.0027,0.000887,3.272627,0.000911,24.580614,0.001364,-933.285706,-820.642944,-924.752197,21.077814,-38.625443,850.824097,850.824097,0.002485,-30.312296,0.011298,-880.766235,1.618738,-852.396912,…,null,-0.155098,null,999.75238,null,-0.004183,null,-0.428273,null,0.000227,null,0.000529,null,-0.001408,null,-905.382935,null,0.000861,null,7.748683,null,999.897156,null,1.146676,null,-997.802979,null,999.992981,null,304.492065,null,42.567852,null,16.287424,null,689.855469,null
4.61374,0.346987,-4.61374,39.584473,0.035124,-0.000201,0.000894,-0.00101,0.399085,0.000211,0.000211,0.000287,0.000287,0.000211,0.000287,-0.002074,0.00218,0.001038,0.001969,0.000751,3.277016,0.000734,24.396032,0.001196,-932.274963,-820.574524,-919.15979,21.406216,-35.163902,848.734741,848.734741,0.000708,-30.312548,0.00984,-882.784851,1.621783,-850.310547,…,null,-0.155174,null,999.748169,null,-0.005132,null,-0.42868,null,0.000224,null,0.000529,null,-0.001787,null,-906.077148,null,0.000823,null,7.750024,null,999.891174,null,1.569156,null,-997.558289,null,999.992737,null,306.457001,null,45.383194,null,16.081768,null,688.799805,null
4.614167,0.347023,-4.614167,39.584934,0.035124,-0.000201,0.000895,-0.001565,0.399496,0.000211,0.000211,0.000287,0.000287,0.000211,0.000287,-0.00296,0.002477,0.00126,0.002266,0.000972,3.275828,0.000904,24.393003,0.001196,-932.278503,-820.552307,-919.160339,21.398781,-35.160927,848.739502,848.739502,0.000918,-30.306437,0.00984,-882.779846,1.615345,-850.308899,…,null,-0.155371,null,999.745911,null,-0.005076,null,-0.427764,null,0.000224,null,0.000389,null,-0.00278,null,-906.075073,null,0.000376,null,7.746992,null,999.886597,null,1.564148,null,-997.567932,null,999.99176,null,306.454529,null,45.376663,null,16.081238,null,688.80011,null
4.61253,0.346467,-4.61253,39.584015,0.035124,-0.000201,0.000747,-0.001328,0.399794,0.000211,0.000211,0.000287,0.000287,0.000211,0.000287,-0.002572,0.002635,0.001697,0.002424,0.001409,3.274771,0.001359,24.394239,0.001736,-932.273682,-820.521362,-919.151978,21.399391,-35.157848,848.746948,848.746948,0.001632,-30.299213,0.009764,-882.770935,1.615503,-850.317627,…,null,-0.155371,null,999.750977,null,-0.005289,null,-0.427713,null,0.000238,null,0.000389,null,-0.002587,null,-906.070312,null,0.000304,null,7.748064,null,999.886108,null,1.553958,null,-997.571106,null,999.992493,null,306.453247,null,45.385536,null,16.083206,null,688.802429,null
4.614835,0.346826,-4.614835,39.586399,0.035124,-0.000219,0.000745,-0.001328,0.399825,0.000211,0

In [46]:
flux.group_by('experiment').agg(pl.all().mean()).with_columns(pl.lit(1)).pivot(
    on = 'experiment',
    index = 'literal',
    values= pl.selectors.exclude('experiment'),
)

literal,10FTHF7GLUtl_Disp,10FTHF7GLUtl_Clus,10FTHF7GLUtm_Disp,10FTHF7GLUtm_Clus,10FTHFtl_Disp,10FTHFtl_Clus,10FTHFtm_Disp,10FTHFtm_Clus,11DOCRTSLtm_Disp,11DOCRTSLtm_Clus,11DOCRTSLtr_Disp,11DOCRTSLtr_Clus,11DOCRTSTRNtm_Disp,11DOCRTSTRNtm_Clus,11DOCRTSTRNtr_Disp,11DOCRTSTRNtr_Clus,13DAMPPOX_Disp,13DAMPPOX_Clus,24_25DHVITD2t_Disp,24_25DHVITD2t_Clus,24_25DHVITD2tm_Disp,24_25DHVITD2tm_Clus,24_25DHVITD3t_Disp,24_25DHVITD3t_Clus,24_25DHVITD3tm_Disp,24_25DHVITD3tm_Clus,24_25VITD2Hm_Disp,24_25VITD2Hm_Clus,24_25VITD3Hm_Disp,24_25VITD3Hm_Clus,24NPHte_Disp,24NPHte_Clus,25HVITD2tin_m_Disp,25HVITD2tin_m_Clus,25HVITD3tin_m_Disp,25HVITD3tin_m_Clus,…,EX_rbl_D[e]_Clus,EX_M01966[e]_Disp,EX_M01966[e]_Clus,EX_M02155[e]_Disp,EX_M02155[e]_Clus,EX_M02837[e]_Disp,EX_M02837[e]_Clus,EX_M01881[e]_Disp,EX_M01881[e]_Clus,EX_M03131[e]_Disp,EX_M03131[e]_Clus,EX_n5m2masn[e]_Disp,EX_n5m2masn[e]_Clus,sink_4abut[l]_Disp,sink_4abut[l]_Clus,DM_4glu56dihdind[c]_Disp,DM_4glu56dihdind[c]_Clus,DM_ind56qn[c]_Disp,DM_ind56qn[c]_Clus,DM_cbl2[m]_Disp,DM_cbl2[m]_Clus,DM_btn[m]_Disp,DM_btn[m]_Clus,DCMPtm_Disp,DCMPtm_Clus,ATPS4mi_Disp,ATPS4mi_Clus,CYOR_u10mi_Disp,CYOR_u10mi_Clus,Htmi_Disp,Htmi_Clus,NADH2_u10mi_Disp,NADH2_u10mi_Clus,CYOOm3i_Disp,CYOOm3i_Clus,literal_Disp,literal_Clus
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,i32
1,7.462157,9.62058,4.703891,3.083415,-7.462157,-9.62058,128.021562,7.125421,2.28608,3.200458,-2.325641,-1.665786,3.765309,3.170503,-3.048093,-1.669585,7.912603,8.298917,2.763118,4.502623,2.763118,4.502623,3.387344,2.812273,3.387344,2.812273,2.763118,4.502623,3.387344,2.812273,-2.903646,-10.489464,8.664126,11.657422,5.733163,5.963211,…,-18.353705,761.50355,894.4809,-96.08405,-11.328055,-258.1458,-68.649881,0.693327,1.130563,1.574862,1.986872,-1.494904,-1.116058,-202.200375,-133.047687,3.235359,6.474861,124.268175,37.144775,928.4764,908.3233,146.8643,116.649375,-702.8065,-849.1523,930.4348,980.1735,523.00375,485.05475,594.15095,625.88205,84.169763,83.132413,471.7992,568.45725,1,1


In [58]:
(
    flux
    .group_by('experiment')
    .agg(pl.all().mean())
    .pivot(
        on='experiment',
        index="dummy_index",
        values=pl.selectors.exclude('experiment'),
    )
)

dummy_index,10FTHF7GLUtl_Clus,10FTHF7GLUtl_Disp,10FTHF7GLUtm_Clus,10FTHF7GLUtm_Disp,10FTHFtl_Clus,10FTHFtl_Disp,10FTHFtm_Clus,10FTHFtm_Disp,11DOCRTSLtm_Clus,11DOCRTSLtm_Disp,11DOCRTSLtr_Clus,11DOCRTSLtr_Disp,11DOCRTSTRNtm_Clus,11DOCRTSTRNtm_Disp,11DOCRTSTRNtr_Clus,11DOCRTSTRNtr_Disp,13DAMPPOX_Clus,13DAMPPOX_Disp,24_25DHVITD2t_Clus,24_25DHVITD2t_Disp,24_25DHVITD2tm_Clus,24_25DHVITD2tm_Disp,24_25DHVITD3t_Clus,24_25DHVITD3t_Disp,24_25DHVITD3tm_Clus,24_25DHVITD3tm_Disp,24_25VITD2Hm_Clus,24_25VITD2Hm_Disp,24_25VITD3Hm_Clus,24_25VITD3Hm_Disp,24NPHte_Clus,24NPHte_Disp,25HVITD2tin_m_Clus,25HVITD2tin_m_Disp,25HVITD3tin_m_Clus,25HVITD3tin_m_Disp,…,EX_rbl_D[e]_Disp,EX_M01966[e]_Clus,EX_M01966[e]_Disp,EX_M02155[e]_Clus,EX_M02155[e]_Disp,EX_M02837[e]_Clus,EX_M02837[e]_Disp,EX_M01881[e]_Clus,EX_M01881[e]_Disp,EX_M03131[e]_Clus,EX_M03131[e]_Disp,EX_n5m2masn[e]_Clus,EX_n5m2masn[e]_Disp,sink_4abut[l]_Clus,sink_4abut[l]_Disp,DM_4glu56dihdind[c]_Clus,DM_4glu56dihdind[c]_Disp,DM_ind56qn[c]_Clus,DM_ind56qn[c]_Disp,DM_cbl2[m]_Clus,DM_cbl2[m]_Disp,DM_btn[m]_Clus,DM_btn[m]_Disp,DCMPtm_Clus,DCMPtm_Disp,ATPS4mi_Clus,ATPS4mi_Disp,CYOR_u10mi_Clus,CYOR_u10mi_Disp,Htmi_Clus,Htmi_Disp,NADH2_u10mi_Clus,NADH2_u10mi_Disp,CYOOm3i_Clus,CYOOm3i_Disp,dummy_index_Clus,dummy_index_Disp
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1.0,9.62058,7.462157,3.083415,4.703891,-9.62058,-7.462157,7.125421,128.021562,3.200458,2.28608,-1.665786,-2.325641,3.170503,3.765309,-1.669585,-3.048093,8.298917,7.912603,4.502623,2.763118,4.502623,2.763118,2.812273,3.387344,2.812273,3.387344,4.502623,2.763118,2.812273,3.387344,-10.489464,-2.903646,11.657422,8.664126,5.963211,5.733163,…,-48.361375,894.4809,761.50355,-11.328055,-96.08405,-68.649881,-258.1458,1.130563,0.693327,1.986872,1.574862,-1.116058,-1.494904,-133.047687,-202.200375,6.474861,3.235359,37.144775,124.268175,908.3233,928.4764,116.649375,146.8643,-849.1523,-702.8065,980.1735,930.4348,485.05475,523.00375,625.88205,594.15095,83.132413,84.169763,568.45725,471.7992,1.0,1.0


In [59]:
flux_pd = flux.to_pandas()

In [74]:
flux_pivoted = flux_pd.pivot_table(
    columns='experiment',
    aggfunc='mean',
)
flux_pivoted = pl.from_pandas(flux_pivoted, include_index=True).rename({'None':'Reaction'})
flux_pivoted

Reaction,Clus,Disp
str,f64,f64
"""10FTHF7GLUtl""",9.620584,7.46216
"""10FTHF7GLUtm""",3.083415,4.703892
"""10FTHFtl""",-9.620584,-7.46216
"""10FTHFtm""",7.125422,128.021606
"""11DOCRTSLte""",7.270179,5.589819
…,…,…
"""sink_tetdec2coa[c]""",-25.029755,-8.793982
"""sink_tetdece1coa[c]""",-16.638071,-24.45936
"""sink_tmndnc[c]""",42.993748,-1.691928


In [75]:
subsystem

Reaction,subSystem
str,str
"""10FTHF5GLUtl""","""Transport, lysosomal"""
"""10FTHF5GLUtm""","""Transport, mitochondrial"""
"""10FTHF6GLUtl""","""Transport, lysosomal"""
"""10FTHF6GLUtm""","""Transport, mitochondrial"""
"""10FTHF7GLUtl""","""Transport, lysosomal"""
…,…
"""CYOR_u10mi""","""Oxidative phosphorylation"""
"""Htmi""","""Transport, mitochondrial"""
"""NADH2_u10mi""","""Oxidative phosphorylation"""


In [78]:
flux_pivoted.filter(pl.col('Reaction').is_in(reactions['Reaction'])).join(reactions, on='Reaction')

Reaction,Clus,Disp,fc,pval,subSystem
str,f64,f64,f64,i64,str
"""10FTHFtm""",7.125422,128.021606,0.894553,0,"""Transport, mitochondrial"""
"""34DHOXPEGOX""",48.990303,510.383087,0.824839,0,"""Tyrosine metabolism"""
"""34DHOXPEGt""",48.990303,510.383087,0.824839,0,"""Transport, extracellular"""
"""34DHPHAMT""",4.596138,288.44577,0.968632,0,"""Tyrosine metabolism"""
"""AATAi""",179.793015,-88.57859,-2.942206,0,"""Lysine metabolism"""
…,…,…,…,…,…
"""4HATVACIDthc""",3.956454,-4.899225,-9.393254,0,"""Drug metabolism"""
"""ALLOP1tu""",196.925705,2.265501,-0.977253,0,"""Drug metabolism"""
"""CRVSM24tev""",1.661527,-2.439606,-5.270844,0,"""Drug metabolism"""


In [116]:
def construct_comprehensive_df(flux, reactions, subsystem, ):
    flux_pivoted = flux.to_pandas()
    flux_pivoted = flux_pivoted.pivot_table(
    columns='experiment',
    aggfunc='mean',
    )
    flux_pivoted = pl.from_pandas(flux_pivoted, include_index=True).rename({'None':'Reaction'})
    # flux_pivoted = pl.from_pandas(flux_pivoted, include_index=True)
    # print(flux_pivoted)
    # return flux_pivoted
    df = flux_pivoted.filter(pl.col('Reaction').is_in(reactions['Reaction'])).join(reactions, on='Reaction')
    df = df.join(subsystem, on='Reaction')
    return df

In [117]:
macir_dfs = Path("../../results/sampling_results/MACIR/").glob("*.csv")
macir_flux = pl.read_parquet("../../outputs/sampling/MACIR/common.parquet")

In [118]:
for file in macir_dfs:
    
    print(file.stem)
    
    macir_df = pl.read_csv(file)
    macir_df = construct_comprehensive_df(macir_flux, macir_df, subsystem)
    
    macir_df.write_csv(f"../../results/sampling_results/MACIR/{file.stem}_subsystem.csv")
    
    print("done")

s1c2_vs_s1k2
done
s1k1_vs_s1k1.fc5
done
s1k2_vs_s1k2.fc5
done
s1k_vs_s1k.fc5
done


In [ ]:
"../../results/sampling_results/MACIR"

Reaction,S1C2,S1K,S1K1,S1K1_FC5,S1K2,S1K2_FC5,S1K_FC5,fc,pval,subSystem
str,f32,f32,f32,f32,f32,f32,f32,f64,i64,str
"""10FTHFtm""",25.666485,304.292175,64.600517,102.569901,468.665253,-315.450226,7.111106,-0.954329,0,"""Transport, mitochondrial"""
"""34DHOXPEGOX""",76.539467,-19.411505,249.940048,301.091858,399.652649,124.242134,156.874664,1.282425,0,"""Tyrosine metabolism"""
"""34DHOXPEGt""",76.539467,-19.411505,249.940048,301.091858,399.652649,124.242134,156.874664,1.282425,0,"""Transport, extracellular"""
"""ACETONEt2""",8.782233,7.565148,8.959544,13.598681,-208.842422,5.038768,-298.931976,-1.051929,0,"""Transport, extracellular"""
"""ACETONEt2m""",8.782233,7.565148,8.959544,13.598681,-208.842422,5.038768,-298.931976,-1.051929,0,"""Transport, mitochondrial"""
…,…,…,…,…,…,…,…,…,…,…
"""ATVLACitr""",-3.780172,2.890864,-3.259059,-2.788948,-16.622206,-2.605255,0.164407,-0.892378,0,"""Drug metabolism"""
"""EX_caproic[e]""",30.045494,-3.238559,-3.330709,-2.627554,-3.073087,-7.976877,13.338968,1.641273,0,"""Exchange/demand reaction"""
"""EX_M00260[e]""",-0.559139,0.622628,0.322193,-0.347404,0.442402,0.395344,-1.125161,-3.477961,0,"""Exchange/demand reaction"""


# Ignore for now

In [15]:
m1 = load_matlab_model("../../outputs/TCGA-arid1a-avg/Truncating.mat")
m2 = load_matlab_model("../../outputs/TCGA-arid1a-avg/WT.mat")

m2

No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, g, i, l, m, n, r, x
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, g, i, l, m, n, r, x


Name,model
Memory address,23d6879bb60
Number of metabolites,4515
Number of reactions,7377
Number of genes,2248
Number of groups,97
Objective expression,1.0*biomass_maintenance - 1.0*biomass_maintenance_reverse_95d2f
Compartments,"c, l, m, r, e, x, g, n, i"


In [21]:
m1

Name,model
Memory address,23b87e763c0
Number of metabolites,4494
Number of reactions,7309
Number of genes,2248
Number of groups,96
Objective expression,1.0*biomass_maintenance - 1.0*biomass_maintenance_reverse_95d2f
Compartments,"c, l, m, r, e, x, g, n, i"


In [16]:
for const in m1.constraints:
    print(const)
    
    break

10fthf5glu[c]: 0 <= 1.0*FPGS7 - 1.0*FPGS7_reverse_5df9f - 1.0*FPGS8 + 1.0*FPGS8_reverse_d2fc1 <= 0


In [17]:
res1 = constraint_matrices(m1)
res2 = constraint_matrices(m2)

In [22]:
res1.equalities

array([[ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       ...,
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  1.,  4., -4.]], shape=(4494, 14618))

In [23]:
res2.equalities

array([[ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       ...,
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  1.,  4., -4.]], shape=(4515, 14754))

In [24]:
print(np.linalg.cond(np.atleast_2d(res1.equalities)))
print(np.linalg.cond(np.atleast_2d(res2.equalities)))


5.6973396258840925e+31
6.293515347649145e+31


In [ ]:
np.linalg.matrix_rank(res1.equalities)

In [12]:
np.any([True, False])

np.True_

In [11]:
np.any(np.isnan(results.equalities))

np.False_

In [14]:
moodel.tolerance

1e-07

In [7]:
np.linalg.svd(results.equalities)

LinAlgError: SVD did not converge

In [8]:
from scipy.linalg import svd

In [9]:
svd(results.equalities)

LinAlgError: SVD did not converge

In [10]:
results.equalities.shape

(4526, 14792)

In [11]:
(np.abs(results.equalities) == np.inf).sum()

0

In [8]:
df = pl.read_csv("../../results/sampling_results/clus_vs_disp/clus_vs_disp_redo.csv", infer_schema_length=None)
df.filter(pl.col('Clus').is_null())

Reaction,fc,pval,Clus,Disp,subSystem
str,f64,f64,str,str,str


In [13]:
df = df.with_columns(pl.col('Clus').cast(pl.Float32, strict=False), pl.col('Disp').cast(pl.Float32, strict=False))

In [18]:
df.filter((pl.col('Disp').is_null()) | (pl.col('fc') > 1))

Reaction,fc,pval,Clus,Disp,subSystem
str,f64,f64,f32,f32,str
"""2DR1PP""",-1.0,-12.32,4.102718,null,"""Pyrimidine catabolism"""
"""2HBO""",1.133483,0.0,-175.056229,10.952519,"""Propanoate metabolism"""
"""2HBt2""",1.716522,0.0,-251.18692,66.254189,"""Transport, extracellular"""
"""3DPHBH2""",-1.0,-12.364964,0.437581,null,"""Ubiquinone synthesis"""
"""ACGALK""",-1.0,-12.410256,14.226833,null,"""Aminosugar metabolism"""
…,…,…,…,…,…
"""EX_M02745[e]""",1.746905,0.0,-1.417648,5.213706,"""Exchange/demand reaction"""
"""sink_asp_L[c]""",1.0,-25.666667,-745.991577,null,"""Exchange/demand reaction"""
"""sink_ser_L[c]""",1.0,-25.862595,-741.114807,null,"""Exchange/demand reaction"""
